# IBM AML Benchmark — Multi-Dataset (Small)
HI-Small + LI-Small · Dirichlet alpha sweep {0.05, 0.1, 0.5}

**FL (6):** FedAvg · FedProx · FedNova · PersFL · SCAFFOLD · FedAvgM  
**ML (5):** XGBoost · LightGBM · CatBoost · Random Forest · Logistic Regression  
**MoE (7):** Static · Performance · Confidence · TypologyAware · ConfidenceAware · ExpertChoice · MultiGate

**Notebook 1 of 2** — Small datasets, all 3 alphas. Estimated runtime: 4–6 h on CPU.


In [ ]:
!pip install lightgbm xgboost imbalanced-learn catboost -q

In [ ]:
from imblearn.over_sampling import SMOTE

print("Running on CPU (no GPU accelerator)")

import os, gc, time, warnings, random
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, fbeta_score
)

import xgboost as xgb
import lightgbm as lgb

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("WARNING: catboost not available")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print("All imports OK")
print(f"  CatBoost available: {HAS_CATBOOST}")

T0  = time.time()
OUT = '/kaggle/working' if os.path.exists('/kaggle') else '.'
def elapsed(): return f"{(time.time()-T0)/60:.1f}m"
def flush():
    gc.collect()

# ================================================================
# CONFIG — NB1: HI-Small + LI-Small, all 3 alphas
# ================================================================
N_BANKS   = 4
ALPHAS    = [0.05, 0.1, 0.5]

DATASETS  = ['HI-Small', 'LI-Small']   # datasets to run in this notebook
IBM_FILES = {
    'HI-Small': 'HI-Small_Trans.csv',
    'LI-Small': 'LI-Small_Trans.csv',
}

# FL strategies (original 4 + 2 new)
FL_STRATEGIES = ['fedavg', 'fedprox', 'fednova', 'persfl', 'scaffold', 'fedavgm']

FL_ROUNDS      = 20
LOCAL_EPOCHS   = 5
MLP_CAP        = 100_000
LR             = 0.001
FEDPROX_MU     = 0.01
SCAFFOLD_LR    = 0.01          # server learning rate for SCAFFOLD correction
FEDAVGM_BETA   = 0.9           # server momentum coefficient
TEST_FRAC      = 0.15
VAL_FRAC       = 0.15
MIN_FRAUD      = 5
THRESH_DEFAULT = 0.5
TUNE_THRESHOLD = True
THRESH_GRID    = np.arange(0.05, 0.95, 0.05)
AP_KS          = (50, 100, 200)

IBM_HIGH_RISK_BANKS = {11, 22, 33, 44, 55, 66}
IBM_PAY_MAP = {
    'ACH': 'Bank Transfer', 'Wire': 'Wire', 'Cheque': 'Cheque',
    'Credit Card': 'Credit Card', 'Debit Card': 'Debit Card',
    'Cash': 'Cash', 'Bitcoin': 'Bitcoin', 'Reinvestment': 'Reinvestment',
}
HIGH_RISK_COUNTRIES = {'mexico', 'turkey', 'morocco', 'uae', 'iran',
                       'nigeria', 'russia', 'venezuela', 'north korea'}


In [ ]:
def find_file(filename):
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f == filename:
                return os.path.join(root, f)
    return None

def load_dataset(ds_name):
    """Load and return raw DataFrame for the given dataset name."""
    fname = IBM_FILES[ds_name]
    path  = find_file(fname)
    if not path:
        raise FileNotFoundError(f"{fname} not found under /kaggle/input")
    print(f"Loading {ds_name}: {path}")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"  {len(df):,} rows | fraud: {df['Is Laundering'].sum():,} ({df['Is Laundering'].mean()*100:.4f}%)")
    return df


def preprocess_ibm(df):
    y   = df['Is Laundering'].astype(int).values
    src = np.full(len(df), 'ibm', dtype=object)

    pay_fmt = df['Payment Format'].astype(str).fillna('Unknown')
    typ = np.where(y == 1, ('IBM_' + pay_fmt).values, 'IBM_Normal')

    ts = pd.to_datetime(df['Timestamp'], errors='coerce')
    valid_mask = ts.notna().values
    if valid_mask.sum() < len(df) * 0.5:
        t_col = np.arange(len(df), dtype=np.int64)
    else:
        ts_int = ts.astype('int64', copy=False).values
        t_col = np.where(valid_mask, ts_int, 0).astype(np.int64)

    amt = df['Amount Paid'].fillna(0).abs()
    amt_log   = np.log1p(amt).values.astype(np.float32)
    amt_round = (amt % 1000 == 0).astype(int).values.astype(np.float32)
    amt_thresh= (amt < 10000).astype(int).values.astype(np.float32)

    hours = ts.dt.hour.fillna(0).values
    hr_sin = np.sin(2 * np.pi * hours / 24).astype(np.float32)
    hr_cos = np.cos(2 * np.pi * hours / 24).astype(np.float32)

    pay_mapped = pay_fmt.map(lambda x: IBM_PAY_MAP.get(x, x))
    le_p = LabelEncoder()
    pt_enc = le_p.fit_transform(pay_mapped).astype(np.float32)

    fb_int = df['From Bank'].fillna(0).astype(int)
    tb_int = df['To Bank'].fillna(0).astype(int)
    sl_risk = fb_int.isin(IBM_HIGH_RISK_BANKS).astype(int).values.astype(np.float32)
    rl_risk = tb_int.isin(IBM_HIGH_RISK_BANKS).astype(int).values.astype(np.float32)
    cross_border = (df['From Bank'].fillna(-1) != df['To Bank'].fillna(-2)).astype(int).values.astype(np.float32)
    curr_mm = (df['Payment Currency'].astype(str) !=
               df['Receiving Currency'].astype(str)).astype(int).values.astype(np.float32)

    X = np.stack([
        amt_log, amt_round, amt_thresh,
        hr_sin, hr_cos, pt_enc,
        sl_risk, rl_risk, cross_border, curr_mm,
    ], axis=1).astype(np.float32)

    print(f"  IBM AML features: {X.shape}")
    return X, y, typ, src, t_col


In [ ]:
def _allocate_fraud(n_fraud, n_banks, alpha):
    props = np.random.dirichlet([alpha] * n_banks)
    fcnts = (props * n_fraud).astype(int)
    diff = n_fraud - fcnts.sum()
    if diff > 0:
        for _ in range(diff): fcnts[np.argmin(fcnts)] += 1
    elif diff < 0:
        for _ in range(-diff): fcnts[np.argmax(fcnts)] -= 1
    assert fcnts.sum() == n_fraud
    return fcnts

def partition_dataset(X, y, typ, t_col, n_banks, alpha):
    np.random.seed(42)
    fraud_idx = np.where(y == 1)[0]; legit_idx = np.where(y == 0)[0]
    np.random.shuffle(fraud_idx); np.random.shuffle(legit_idx)
    fcnts = _allocate_fraud(len(fraud_idx), n_banks, alpha)
    lper = len(legit_idx) // n_banks
    banks = []; fp = lp = 0
    for i in range(n_banks):
        ln = lper if i < n_banks - 1 else len(legit_idx) - lp
        idx = np.concatenate([fraud_idx[fp:fp+fcnts[i]], legit_idx[lp:lp+ln]])
        name = f'IBM-Bank{i}'
        banks.append(_split(i, X[idx], y[idx], typ[idx], t_col[idx], name))
        fp += fcnts[i]; lp += ln
    print(f"Partition [IBM] {n_banks} banks | alpha={alpha}:")
    for b in banks:
        print(f"  {b['name']:20s}: {b['n_total']:>10,} txns | "
              f"{b['n_fraud']:>6,} fraud ({b['fraud_frac']*100:.3f}%) | "
              f"train={b['n_train_fraud']} val={b['n_val_fraud']} test={b['n_test_fraud']} fraud")
    return banks

def _split(bid, X, y, typ, t, name):
    order = np.argsort(t, kind='stable')
    X, y, typ = X[order], y[order], typ[order]
    n = len(X)
    n_train = max(int(n * (1 - TEST_FRAC - VAL_FRAC)), 1)
    n_val   = max(int(n * VAL_FRAC), 1)
    if n_train + n_val >= n: n_train = max(n - 2, 1); n_val = 1
    Xtr, ytr, ttr = X[:n_train],              y[:n_train],              typ[:n_train]
    Xvl, yvl, tvl = X[n_train:n_train+n_val], y[n_train:n_train+n_val], typ[n_train:n_train+n_val]
    Xte, yte, tte = X[n_train+n_val:],        y[n_train+n_val:],        typ[n_train+n_val:]
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr).astype(np.float32)
    Xvl = sc.transform(Xvl).astype(np.float32)
    Xte = sc.transform(Xte).astype(np.float32)
    return dict(
        id=bid, name=name, source='ibm',
        X_train=Xtr, y_train=ytr, typ_train=ttr,
        X_val=Xvl,   y_val=yvl,   typ_val=tvl,
        X_test=Xte,  y_test=yte,  typ_test=tte,
        n_fraud=int(y.sum()), n_total=len(y), fraud_frac=float(y.mean()),
        n_train_fraud=int(ytr.sum()), n_val_fraud=int(yvl.sum()), n_test_fraud=int(yte.sum()),
    )

def safe_smote(X, y):
    if y.sum() < 5 or len(X) < 20: return X, y
    try:
        k = min(5, int(y.sum()) - 1)
        Xs, ys = SMOTE(random_state=42, k_neighbors=k).fit_resample(X, y)
        return Xs.astype(np.float32), ys.astype(np.int64)
    except Exception:
        return X, y

def get_mlp_data(bank):
    Xtr, ytr = bank['X_train'], bank['y_train']
    if len(Xtr) > MLP_CAP:
        fi = np.where(ytr == 1)[0]; li = np.where(ytr == 0)[0]
        nf = min(len(fi), MLP_CAP // 10); nl = min(len(li), MLP_CAP - nf)
        np.random.shuffle(fi); np.random.shuffle(li)
        idx = np.concatenate([fi[:nf], li[:nl]]); np.random.shuffle(idx)
        Xtr, ytr = Xtr[idx], ytr[idx]
    return safe_smote(Xtr, ytr)


In [ ]:
# ================================================================
# MLP helpers
# ================================================================
def make_mlp():
    return MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu',
        solver='adam', learning_rate_init=LR,
        max_iter=LOCAL_EPOCHS, random_state=42,
        warm_start=False, tol=1e-4)

def get_weights(m):
    return [c.copy() for c in m.coefs_] + [i.copy() for i in m.intercepts_]

def set_weights(m, w):
    n = len(m.coefs_)
    m.coefs_ = [x.copy() for x in w[:n]]
    m.intercepts_ = [x.copy() for x in w[n:]]

def init_mlp(X, y):
    if 0 not in y: X,y = np.vstack([X,X[0:1]]), np.append(y,0)
    if 1 not in y: X,y = np.vstack([X,X[0:1]]), np.append(y,1)
    m = make_mlp(); m.fit(X, y); return m

def clone_mlp(template, w):
    m = make_mlp()
    tiny = np.zeros((2, template.coefs_[0].shape[0]), dtype=np.float32)
    try: m.fit(tiny, np.array([0,1]))
    except: pass
    set_weights(m, w); return m

def mlp_proba(m, X):
    try:    return m.predict_proba(X)[:,1]
    except: return np.full(len(X), 0.5)

def w_zeros_like(w):
    return [np.zeros_like(x) for x in w]

def w_add(a, b):
    return [x + y for x, y in zip(a, b)]

def w_scale(a, s):
    return [x * s for x in a]

def w_sub(a, b):
    return [x - y for x, y in zip(a, b)]

# ================================================================
# FL ALGORITHMS
# ================================================================
def fedavg_agg(gw, wlist, sizes):
    total = sum(sizes)
    return [sum((s/total)*wl[i] for wl,s in zip(wlist,sizes))
            for i in range(len(gw))]

def run_fedavg(banks, gw, tmpl):
    wl=[]; sz=[]
    for b in banks:
        Xtr,ytr = get_mlp_data(b)
        if len(Xtr) == 0: continue
        lm = clone_mlp(tmpl, gw); lm.fit(Xtr, ytr)
        wl.append(get_weights(lm)); sz.append(len(Xtr)); del lm; flush()
    return fedavg_agg(gw, wl, sz) if wl else gw

def run_fedprox(banks, gw, tmpl):
    wl=[]; sz=[]
    for b in banks:
        Xtr,ytr = get_mlp_data(b)
        if len(Xtr) == 0: continue
        lm = clone_mlp(tmpl, gw); lm.fit(Xtr, ytr)
        local_w = get_weights(lm)
        prox = [lw - FEDPROX_MU*(lw - g) for lw,g in zip(local_w, gw)]
        wl.append(prox); sz.append(len(Xtr)); del lm; flush()
    return fedavg_agg(gw, wl, sz) if wl else gw

def run_fednova(banks, gw, tmpl):
    dl=[]; sz=[]; st=[]
    for b in banks:
        Xtr,ytr = get_mlp_data(b)
        if len(Xtr) == 0: continue
        lm = clone_mlp(tmpl, gw); lm.fit(Xtr, ytr)
        lw = get_weights(lm)
        delta = [l - g for l,g in zip(lw, gw)]
        tau = LOCAL_EPOCHS * max(len(Xtr)//32, 1)
        dl.append(delta); sz.append(len(Xtr)); st.append(tau); del lm; flush()
    if not dl: return gw
    total = sum(sz); tau_g = sum(t*s/total for t,s in zip(st,sz))
    return [gw[i] + tau_g*sum((s/total)*(d[i]/max(t,1))
            for d,s,t in zip(dl,sz,st)) for i in range(len(gw))]

def run_persfl(banks, gw, tmpl, bw):
    BLEND=0.5; wl=[]; sz=[]
    for b in banks:
        bid=b['id']; Xtr,ytr=get_mlp_data(b)
        if len(Xtr) == 0: continue
        lm = clone_mlp(tmpl, bw.get(bid, gw)); lm.fit(Xtr, ytr)
        lw = get_weights(lm)
        bw[bid] = [BLEND*l+(1-BLEND)*g for l,g in zip(lw, gw)]
        wl.append(lw); sz.append(len(Xtr)); del lm; flush()
    return (fedavg_agg(gw, wl, sz) if wl else gw), bw

# ── SCAFFOLD ──────────────────────────────────────────────────────
# Simplified SCAFFOLD: each client maintains a control variate c_i.
# Global control variate c = mean(c_i). Client update corrects drift.
# c_i update: c_i_new = c_i - c + (1/K*lr) * (w_global - w_local)
def run_scaffold(banks, gw, tmpl, c_global, c_clients):
    """
    c_global : list of arrays (same shape as gw) — global control variate
    c_clients: dict bid -> list of arrays — per-client control variates
    Returns: new_gw, new_c_global, new_c_clients
    """
    delta_ws = []; delta_cs = []; sz = []
    K = LOCAL_EPOCHS

    for b in banks:
        bid = b['id']; Xtr, ytr = get_mlp_data(b)
        if len(Xtr) == 0: continue

        c_i = c_clients.get(bid, w_zeros_like(gw))
        lm  = clone_mlp(tmpl, gw)

        # Simulate K local SGD steps with SCAFFOLD correction
        # (We approximate by: train normally, then apply control-variate correction)
        lm.fit(Xtr, ytr)
        lw = get_weights(lm)

        # SCAFFOLD correction: shift local update by (c_global - c_i)
        correction = w_sub(c_global, c_i)
        lw_corrected = w_add(lw, w_scale(correction, SCAFFOLD_LR))

        # Update client control variate
        # c_i_new = c_i - c_global + (1/(K*lr)) * (gw - lw_corrected)
        c_i_new = w_add(
            w_sub(c_i, c_global),
            w_scale(w_sub(gw, lw_corrected), 1.0 / max(K * LR, 1e-8))
        )
        # Clip to avoid explosion
        c_i_new = [np.clip(x, -10, 10) for x in c_i_new]

        delta_ws.append(w_sub(lw_corrected, gw))
        delta_cs.append(w_sub(c_i_new, c_i))
        c_clients[bid] = c_i_new
        sz.append(len(Xtr)); del lm; flush()

    if not delta_ws:
        return gw, c_global, c_clients

    total = sum(sz)
    # Aggregate: gw += (1/n_clients) * sum(delta_w)
    n = len(delta_ws)
    new_gw = [gw[i] + sum(d[i] for d in delta_ws) / n for i in range(len(gw))]
    # Update global control variate
    new_c_global = [c_global[i] + sum(d[i] for d in delta_cs) / len(banks)
                    for i in range(len(gw))]
    return new_gw, new_c_global, c_clients

# ── FedAvgM (Server Momentum) ─────────────────────────────────────
def run_fedavgm(banks, gw, tmpl, momentum_buf):
    """
    FedAvg with server-side momentum (Polyak-style).
    momentum_buf: list of arrays (same shape as gw), initialized to zeros.
    Returns: new_gw, new_momentum_buf
    """
    wl=[]; sz=[]
    for b in banks:
        Xtr,ytr = get_mlp_data(b)
        if len(Xtr) == 0: continue
        lm = clone_mlp(tmpl, gw); lm.fit(Xtr, ytr)
        wl.append(get_weights(lm)); sz.append(len(Xtr)); del lm; flush()
    if not wl:
        return gw, momentum_buf

    agg_w = fedavg_agg(gw, wl, sz)
    pseudo_grad = w_sub(gw, agg_w)  # negative gradient direction

    # momentum update: buf = beta * buf + grad; w = w - buf
    new_buf = [FEDAVGM_BETA * m + g for m, g in zip(momentum_buf, pseudo_grad)]
    new_gw  = [w - b for w, b in zip(gw, new_buf)]
    return new_gw, new_buf

# ================================================================
# UNIFIED run_fl
# ================================================================
def run_fl(banks, strategy, n_feat):
    print(f"  [{strategy.upper()}] {FL_ROUNDS} rounds", end='  ', flush=True)
    init_X = np.vstack([b['X_train'][:50] for b in banks])
    init_y = np.concatenate([b['y_train'][:50] for b in banks])
    tmpl = init_mlp(init_X, init_y)
    gw = get_weights(tmpl); bw = {}; hist = []

    # Extra state for new algorithms
    c_global  = w_zeros_like(gw)   # SCAFFOLD global control variate
    c_clients = {}                  # SCAFFOLD per-client control variates
    momentum_buf = w_zeros_like(gw) # FedAvgM momentum buffer

    for rnd in range(1, FL_ROUNDS + 1):
        if   strategy == 'fedavg':   gw = run_fedavg(banks, gw, tmpl)
        elif strategy == 'fedprox':  gw = run_fedprox(banks, gw, tmpl)
        elif strategy == 'fednova':  gw = run_fednova(banks, gw, tmpl)
        elif strategy == 'persfl':   gw, bw = run_persfl(banks, gw, tmpl, bw)
        elif strategy == 'scaffold':
            gw, c_global, c_clients = run_scaffold(banks, gw, tmpl, c_global, c_clients)
        elif strategy == 'fedavgm':
            gw, momentum_buf = run_fedavgm(banks, gw, tmpl, momentum_buf)

        if rnd % 5 == 0 or rnd == FL_ROUNDS:
            print(f"{rnd}", end=' ', flush=True)
            gm = clone_mlp(tmpl, gw)
            for b in banks:
                yt = b['y_test']
                if strategy == 'persfl' and b['id'] in bw:
                    em = clone_mlp(tmpl, bw[b['id']])
                else:
                    em = gm
                probs = mlp_proba(em, b['X_test'])
                preds = (probs >= THRESH_DEFAULT).astype(int)
                f1_val    = f1_score(yt, preds, zero_division=0) if yt.sum() > 0 else 0.
                auprc_val = average_precision_score(yt, probs)   if yt.sum() > 0 else 0.
                hist.append(dict(round=rnd, bank_id=b['id'], bank_name=b['name'],
                    strategy=strategy, source='ibm',
                    f1=f1_val, auprc=auprc_val, n_test_fraud=int(yt.sum())))
        flush()
    print()
    return clone_mlp(tmpl, gw), bw, hist


In [ ]:
def tune_threshold(y_val, probs_val, default=THRESH_DEFAULT):
    if y_val.sum() == 0 or y_val.sum() == len(y_val):
        return default
    best_t, best_f2 = default, -1
    for t in THRESH_GRID:
        preds = (probs_val >= t).astype(int)
        prec = precision_score(y_val, preds, zero_division=0)
        rec  = recall_score(y_val, preds, zero_division=0)
        d = 4*prec + rec
        f2 = 5*prec*rec/d if d > 0 else 0
        if f2 > best_f2: best_f2, best_t = f2, t
    return best_t

# ================================================================
# EXPERT CLASSES
# ================================================================
class XGBExpert:
    name = 'xgb'
    def __init__(self, pw):
        self.m = xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            scale_pos_weight=pw, tree_method='hist',
            use_label_encoder=False, eval_metric='aucpr',
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0)
        self.trained = False
    def fit(self, X, y):
        self.m.fit(X, y); self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:,1] if self.trained else np.full(len(X), 0.5)

class LGBMExpert:
    name = 'lgbm'
    def __init__(self, pw):
        self.m = lgb.LGBMClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            scale_pos_weight=pw, device='cpu',
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbose=-1)
        self.trained = False
    def fit(self, X, y):
        self.m.fit(X, y); self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:,1] if self.trained else np.full(len(X), 0.5)

class CatBoostExpert:
    name = 'catboost'
    def __init__(self, pw):
        try:
            self.m = CatBoostClassifier(iterations=200, depth=6,
                scale_pos_weight=pw, eval_metric='PRAUC', verbose=0, random_seed=42,
                subsample=0.8, bagging_temperature=0.5, bootstrap_type='Bernoulli')
        except Exception:
            self.m = CatBoostClassifier(iterations=200, depth=6,
                scale_pos_weight=pw, eval_metric='PRAUC', verbose=0, random_seed=42)
        self.trained = False
    def fit(self, X, y):
        try:
            self.m.fit(X, y)
        except Exception:
            self.m = CatBoostClassifier(iterations=200, depth=6,
                scale_pos_weight=float(((y==0).sum()/max((y==1).sum(),1))),
                eval_metric='PRAUC', verbose=0, random_seed=42)
            self.m.fit(X, y)
        self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:,1] if self.trained else np.full(len(X), 0.5)

# ── NEW: Random Forest ───────────────────────────────────────────
class RFExpert:
    """Random Forest — benefits from cuml.accel GPU acceleration."""
    name = 'rf'
    def __init__(self, pw):
        # class_weight maps to imbalance ratio
        self.m = RandomForestClassifier(
            n_estimators=200, max_depth=10,
            class_weight={0: 1, 1: int(pw)},
            n_jobs=-1, random_state=42)
        self.trained = False
    def fit(self, X, y):
        self.m.fit(X, y); self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:,1] if self.trained else np.full(len(X), 0.5)

# ── NEW: Logistic Regression (calibrated baseline) ───────────────
class LRExpert:
    """Logistic Regression — calibrated probabilistic baseline."""
    name = 'lr'
    def __init__(self, pw):
        self.m = LogisticRegression(
            C=1.0, class_weight={0: 1, 1: int(max(pw, 1))},
            solver='lbfgs', max_iter=1000, random_state=42, n_jobs=-1)
        self.trained = False
    def fit(self, X, y):
        self.m.fit(X, y); self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:,1] if self.trained else np.full(len(X), 0.5)

EXPERT_CLASSES = [XGBExpert, LGBMExpert, RFExpert, LRExpert]
if HAS_CATBOOST:
    EXPERT_CLASSES.insert(2, CatBoostExpert)  # insert before RF
EXPERT_NAMES = [c.name for c in EXPERT_CLASSES]
print(f"Expert models: {EXPERT_NAMES}")

def train_experts(banks):
    experts = {}
    for b in banks:
        bid = b['id']; experts[bid] = {}
        Xtr, ytr = safe_smote(b['X_train'], b['y_train'])
        n0 = (ytr==0).sum(); n1 = max((ytr==1).sum(), 1); pw = float(n0/n1)
        if len(Xtr) == 0 or ytr.sum() == 0:
            for Cls in EXPERT_CLASSES: experts[bid][Cls.name] = None
            print(f"  {b['name']:20s} SKIPPED: no fraud in training set")
            continue
        for Cls in EXPERT_CLASSES:
            try:
                m = Cls(pw); m.fit(Xtr, ytr)
                p = m.predict_proba(b['X_test'])
                pd_ = (p >= THRESH_DEFAULT).astype(int)
                if b['y_test'].sum() > 0:
                    f1 = f1_score(b['y_test'], pd_, zero_division=0)
                    ap = average_precision_score(b['y_test'], p)
                    print(f"  {b['name']:20s} {m.name.upper():10s}: F1={f1:.4f}  AUPRC={ap:.4f}")
                else:
                    tn = int(((b['y_test']==0)&(pd_==0)).sum())
                    fp = int(((b['y_test']==0)&(pd_==1)).sum())
                    spec = tn/max(tn+fp,1); fpr = fp/max(tn+fp,1)
                    print(f"  {b['name']:20s} {m.name.upper():10s}: (no test fraud) "
                          f"Specificity={spec:.4f} FPR={fpr:.4f}")
                experts[bid][m.name] = m
            except Exception as e:
                print(f"  {b['name']:20s} {Cls.name.upper():10s}: FAILED ({e})")
                experts[bid][Cls.name] = None
    return experts


In [ ]:
# ================================================================
# MoE GATES
# ================================================================
class StaticGate:
    name='moe_static'
    def get_weights(self,n,avail,**kw):
        w=np.where(avail,1.,0.); return w/w.sum() if w.sum()>0 else w
    def update(self,*a,**k): pass

class PerformanceGate:
    name='moe_performance'
    def __init__(self): self._w={}
    def update(self,bid,vf1s,avail,**kw):
        s=np.where(avail,np.clip(vf1s,0,None),0.)
        self._w[bid]=s/s.sum() if s.sum()>0 else np.ones(len(s))/len(s)
    def get_weights(self,n,avail,bid=None,**kw):
        return self._w.get(bid,np.ones(n)/n)

class ConfidenceGate:
    name='moe_confidence'
    def __init__(self): self._w={}
    def update(self,bid,val_probs_list,avail,**kw):
        confs=[]
        for i,p in enumerate(val_probs_list):
            if avail[i] and p is not None and len(p)>0:
                confs.append(float(np.mean(np.abs(p - 0.5) * 2)))
            else:
                confs.append(0.)
        s=np.array(confs)
        self._w[bid]=s/s.sum() if s.sum()>0 else np.ones(len(s))/len(s)
    def get_weights(self,n,avail,bid=None,**kw):
        return self._w.get(bid,np.ones(n)/n)

class TypologyAwareGate:
    name='moe_typology_aware'
    def __init__(self):
        self._tbl=defaultdict(lambda:defaultdict(dict)); self._fb={}
    def update(self,bid,typ_f1s,avail,val_f1s=None,**kw):
        for ei,(tf,av) in enumerate(zip(typ_f1s,avail)):
            if av: self._tbl[bid][ei]=tf
        if val_f1s is not None:
            s=np.where(avail,np.clip(val_f1s,0,None),0.)
            self._fb[bid]=s/s.sum() if s.sum()>0 else np.ones(len(s))/len(s)
    def get_weights(self,n,avail,bid=None,txn_typ=None,**kw):
        if txn_typ is None or bid not in self._tbl or txn_typ=='IBM_Normal':
            return self._fb.get(bid,np.ones(n)/n)
        w=np.zeros(n)
        for ei in range(n):
            if avail[ei] and ei in self._tbl[bid]:
                w[ei]=self._tbl[bid][ei].get(txn_typ,0.)
        return w/w.sum() if w.sum()>0 else self._fb.get(bid,np.ones(n)/n)

# ================================================================
# NEW MoE GATES (theoretically motivated)
# ================================================================

class ConfidenceAwareGate:
    """
    Confidence-Aware Gate (ref: arXiv 2410.09069)
    Routes by prediction uncertainty: low-confidence predictions → spread
    weight across more experts. High-confidence predictions → concentrate
    weight on the most confident expert. Handles class imbalance by giving
    uncertain (likely rare-fraud) samples more careful expert assignment.

    Uncertainty = 1 - |p - 0.5| * 2  (0 = maximally confident, 1 = maximally uncertain)
    Weight for expert i ∝ confidence_i * (1 - uncertainty_i + epsilon)
    """
    name = 'moe_confidence_aware'

    def __init__(self, temp=2.0, min_eps=1e-3):
        self._w  = {}       # per-bank normalised weights
        self._unc = {}      # per-bank per-expert mean uncertainty on val set
        self.temp = temp    # temperature for softmax sharpening
        self.min_eps = min_eps

    def update(self, bid, val_probs_list, avail, **kw):
        """
        val_probs_list: list of 1-D prob arrays (one per expert) on val set.
        For each expert compute mean confidence (1 - uncertainty).
        High confidence → higher routing weight.
        """
        confidences = []
        for i, p in enumerate(val_probs_list):
            if avail[i] and p is not None and len(p) > 0:
                uncertainty = 1.0 - np.abs(p - 0.5) * 2        # per-sample
                # Focus uncertainty on the fraud-score region (p > 0.3)
                high_risk_mask = p > 0.3
                if high_risk_mask.sum() > 0:
                    # Mean uncertainty in the high-risk region (harder to get right)
                    unc_hr = float(np.mean(uncertainty[high_risk_mask]))
                else:
                    unc_hr = float(np.mean(uncertainty))
                conf = 1.0 - unc_hr + self.min_eps
                confidences.append(conf)
            else:
                confidences.append(0.0)
        self._unc[bid] = confidences

        # Temperature-scaled softmax over log-confidence
        arr = np.array(confidences, dtype=np.float64)
        arr = np.where(avail, arr, 0.0)
        if arr.sum() > 0:
            log_w = np.log(arr + self.min_eps) * self.temp
            log_w = log_w - log_w.max()         # numerical stability
            exp_w = np.exp(log_w) * (arr > 0)
            self._w[bid] = exp_w / exp_w.sum()
        else:
            self._w[bid] = np.ones(len(avail)) / len(avail)

    def get_weights(self, n, avail, bid=None, **kw):
        return self._w.get(bid, np.ones(n) / n)


class ExpertChoiceGate:
    """
    Expert-Choice Gate (ref: arXiv 2401.15969, FLEX-MoE 2512.23070)
    Inverted routing: each expert selects its top-K% most confident tokens
    instead of each token selecting an expert. This prevents the majority-class
    (legitimate transactions) from monopolising high-capacity experts and
    ensures fraud samples — which have distinct signal — are captured by
    specialist experts.

    Implementation: on the val set, compute per-expert precision-at-top-K.
    At inference, the expert with highest AP@K for the current score range
    gets the highest weight. Sparse: only top-2 experts get non-zero weight.
    """
    name = 'moe_expert_choice'

    def __init__(self, top_k_frac=0.05, top_n_experts=2):
        self._w       = {}
        self._apk     = {}          # per-bank per-expert AP@K on val
        self.top_k    = top_k_frac  # fraction of val set to score
        self.top_n    = top_n_experts

    def update(self, bid, val_probs_list, avail, val_y=None, **kw):
        """
        val_y: ground-truth labels for validation set.
        For each expert, compute AP@(top_k_frac * N) as a measure of how well
        the expert ranks fraud at the top of its score distribution.
        """
        if val_y is None:
            # Fallback: use confidence as proxy
            self._w[bid] = np.ones(len(avail)) / max(np.sum(avail), 1)
            return

        n = len(val_y)
        k = max(int(self.top_k * n), 5)
        apk_scores = []
        for i, p in enumerate(val_probs_list):
            if avail[i] and p is not None and len(p) == n and val_y.sum() > 0:
                top_idx = np.argsort(p)[::-1][:k]
                ap = float(val_y[top_idx].sum()) / max(k, 1)
                apk_scores.append(ap + 1e-6)   # add epsilon so never 0
            else:
                apk_scores.append(0.0)
        self._apk[bid] = apk_scores

        arr = np.array(apk_scores, dtype=np.float64)
        arr = np.where(avail, arr, 0.0)

        # Sparse: keep only top-N experts, zero out the rest
        if arr.sum() > 0:
            threshold = np.sort(arr)[::-1][min(self.top_n - 1, len(arr) - 1)]
            sparse = np.where(arr >= threshold, arr, 0.0)
            # Tiebreak: if more than top_n non-zero, keep only top_n
            nonzero_idx = np.where(sparse > 0)[0]
            if len(nonzero_idx) > self.top_n:
                keep = nonzero_idx[np.argsort(sparse[nonzero_idx])[::-1][:self.top_n]]
                mask = np.zeros(len(arr))
                mask[keep] = 1.0
                sparse = sparse * mask
            self._w[bid] = sparse / sparse.sum() if sparse.sum() > 0 else np.ones(len(arr)) / len(arr)
        else:
            self._w[bid] = np.ones(len(avail)) / max(np.sum(avail), 1)

    def get_weights(self, n, avail, bid=None, **kw):
        return self._w.get(bid, np.ones(n) / n)


class MultiGateMoE:
    """
    Multi-Gate MoE (ref: Fed-NMoE arXiv 2511.01743)
    Implements the FedGate pattern: shared global experts + per-client local
    gate head. Each client (bank) maintains its own gate weights tuned on
    its local val distribution, but the expert pool is shared (global FL model
    + local ML experts).

    Gate head: a softmax over a learned linear combination of
    (global_performance_score, local_val_f2, confidence_score).
    Three components → one weight vector per client.

    This naturally handles non-IID because each bank's gate head adapts to
    its local fraud distribution without re-training the underlying experts.
    """
    name = 'moe_multi_gate'

    def __init__(self, alpha_global=0.4, alpha_local=0.4, alpha_conf=0.2):
        """
        alpha_*: mixing coefficients for the three gate signals.
        Defaults: balanced between global performance, local F2, confidence.
        """
        self._w         = {}
        self.a_global   = alpha_global
        self.a_local    = alpha_local
        self.a_conf     = alpha_conf

    def update(self, bid, val_probs_list, avail, vf1s=None, val_f1s=None, val_y=None, **kw):
        """
        Combines three signals:
          1. global_score  = val F2 (cross-bank performance)
          2. local_score   = per-bank val F2 (local adaptation signal)
          3. conf_score    = mean confidence on val set
        Final weight = softmax(a_g * g + a_l * l + a_c * c)
        """
        n = len(val_probs_list)
        global_scores = np.array(vf1s if vf1s is not None else [0.]*n, dtype=np.float64)
        local_scores  = np.array(val_f1s if val_f1s is not None else global_scores, dtype=np.float64)

        conf_scores = []
        for i, p in enumerate(val_probs_list):
            if avail[i] and p is not None and len(p) > 0:
                conf_scores.append(float(np.mean(np.abs(p - 0.5) * 2)))
            else:
                conf_scores.append(0.0)
        conf_scores = np.array(conf_scores, dtype=np.float64)

        avail_arr = np.array(avail, dtype=np.float64)

        combined = (self.a_global * np.clip(global_scores, 0, None) +
                    self.a_local  * np.clip(local_scores,  0, None) +
                    self.a_conf   * conf_scores)
        combined = np.where(avail_arr > 0, combined, -1e9)

        # Softmax
        combined = combined - combined[avail_arr > 0].max()
        exp_w = np.exp(combined) * avail_arr
        if exp_w.sum() > 0:
            self._w[bid] = exp_w / exp_w.sum()
        else:
            self._w[bid] = np.where(avail_arr > 0, 1.0 / max(avail_arr.sum(), 1), 0.0)

    def get_weights(self, n, avail, bid=None, **kw):
        return self._w.get(bid, np.ones(n) / n)


def _typ_f1(y_val, probs, typ_val, thresh):
    preds=(probs>=thresh).astype(int); out={}
    for t in np.unique(typ_val):
        if t == 'IBM_Normal': continue
        mask = typ_val == t
        if mask.sum()<2 or y_val[mask].sum()==0: continue
        out[t]=f1_score(y_val[mask],preds[mask],zero_division=0)
    return out

# ================================================================
# METRICS
# ================================================================
TYP_W = {
    'IBM_Wire':2.5,'IBM_ACH':2.0,'IBM_Bitcoin':2.5,
    'IBM_Cheque':1.5,'IBM_Cash':1.5,'IBM_Credit Card':2.0,
    'IBM_Reinvestment':2.0,
}

def compute_metrics(y_true, probs, typ=None, thresh=THRESH_DEFAULT):
    preds = (probs >= thresh).astype(int)
    n_pos = int(y_true.sum())
    tn = int(((y_true==0)&(preds==0)).sum()); fp = int(((y_true==0)&(preds==1)).sum())
    specificity = tn/max(tn+fp,1); fpr = fp/max(tn+fp,1)
    if n_pos == 0:
        apk = {f'ap_at_{k}': 0. for k in AP_KS}
        return {'f1':0.,'precision':0.,'recall':0.,'auprc':0.,
                'mcc':0.,'f2':0.,**apk,
                'typ_coverage':0.,'typ_wf1':0.,
                'specificity':float(specificity),'fpr':float(fpr),
                'false_positives':fp,'n_test_fraud':0,'threshold':thresh}
    f1   = f1_score(y_true,preds,zero_division=0)
    prec = precision_score(y_true,preds,zero_division=0)
    rec  = recall_score(y_true,preds,zero_division=0)
    try:    auprc = average_precision_score(y_true,probs)
    except: auprc = float(y_true.mean())
    mcc = matthews_corrcoef(y_true,preds) if preds.sum()>0 else 0.
    b2=4; d=b2*prec+rec; f2=(1+b2)*prec*rec/d if d>0 else 0.
    sidx = np.argsort(probs)[::-1]
    apk  = {f'ap_at_{k}': float(y_true[sidx[:min(k,len(y_true))]].sum()/
                                 max(min(k,len(y_true)),1)) for k in AP_KS}
    typ_cov=0.; typ_wf1=0.
    if typ is not None:
        laund=[t for t in np.unique(typ) if t!='IBM_Normal']
        if laund:
            typ_cov=sum(1 for t in laund
                if ((typ==t)&(y_true==1)&(preds==1)).sum()>0)/len(laund)
        ws=wt=0.
        for t in np.unique(typ):
            if t=='IBM_Normal': continue
            mask=typ==t
            if mask.sum()<2 or y_true[mask].sum()==0: continue
            f=f1_score(y_true[mask],preds[mask],zero_division=0)
            w=TYP_W.get(t,1.5); ws+=w*f; wt+=w
        typ_wf1=ws/max(wt,1e-8)
    return {'f1':f1,'precision':prec,'recall':rec,'auprc':auprc,
            'mcc':mcc,'f2':f2,**apk,
            'typ_coverage':typ_cov,'typ_wf1':typ_wf1,
            'specificity':float(specificity),'fpr':float(fpr),
            'false_positives':fp,'n_test_fraud':n_pos,'threshold':thresh}

def agg(ml):
    if not ml: return {}
    return {k:round(float(np.mean([m[k] for m in ml])),4) for k in ml[0]}

def fairness(f1s, lf=None):
    r=dict(client_equity=round(float(np.std(f1s)),4),
           worst_bank_f1=round(float(min(f1s)),4),
           best_bank_f1=round(float(max(f1s)),4))
    if lf and len(lf)==len(f1s):
        r['collab_gain']=round(float(np.mean([a-b for a,b in zip(f1s,lf)])),4)
    return r


In [ ]:
def evaluate_all(banks, fl_results, experts):
    rows = []
    n_fl  = len(fl_results)
    n_ml  = len(EXPERT_NAMES)
    n_exp = n_fl + n_ml

    # Local-only F1 baseline per bank
    loc_f1 = {}
    for b in banks:
        bid=b['id']; yt=b['y_test']; best=0.
        for en in EXPERT_NAMES:
            m=experts[bid].get(en)
            if m and m.trained and yt.sum()>0:
                p=m.predict_proba(b['X_test'])
                t_tune=tune_threshold(b['y_val'],m.predict_proba(b['X_val']))
                best=max(best,f1_score(yt,(p>=t_tune).astype(int),zero_division=0))
        loc_f1[bid]=best

    def _row(strat,mtype,bm,lf=None,n_eval=None,n_with_fraud=None,total_fraud=None):
        a=agg(bm); fa=fairness([m['f1'] for m in bm],lf); extra={}
        if n_eval is not None:      extra['n_eval_banks']=n_eval
        if n_with_fraud is not None:extra['n_banks_with_fraud']=n_with_fraud
        if total_fraud is not None: extra['total_test_fraud']=total_fraud
        return {'strategy':strat,'model_type':mtype,**a,**fa,**extra}

    total_fraud=sum(int(b['y_test'].sum()) for b in banks)

    # ── FL ──
    for strat,(fl_model,bw,_) in fl_results.items():
        bm=[]; lf=[]
        for b in banks:
            yt=b['y_test']; tte=b.get('typ_test')
            if strat=='persfl' and b['id'] in bw:
                pv=mlp_proba(bw[b['id']],b['X_val']); pt=mlp_proba(bw[b['id']],b['X_test'])
            else:
                pv=mlp_proba(fl_model,b['X_val']); pt=mlp_proba(fl_model,b['X_test'])
            t_tune=tune_threshold(b['y_val'],pv)
            bm.append(compute_metrics(yt,pt,tte,thresh=t_tune)); lf.append(loc_f1.get(b['id'],0))
        n_wf=sum(1 for b in banks if b['y_test'].sum()>0)
        rows.append(_row(strat,'fl',bm,lf,len(banks),n_wf,total_fraud))

    # ── Local experts ──
    for en in EXPERT_NAMES:
        bm=[]
        for b in banks:
            yt=b['y_test']; tte=b.get('typ_test'); m=experts[b['id']].get(en)
            if m and m.trained:
                pv=m.predict_proba(b['X_val']); pt=m.predict_proba(b['X_test'])
                t_tune=tune_threshold(b['y_val'],pv)
            else:
                pt=np.full(len(yt),0.5); t_tune=THRESH_DEFAULT
            bm.append(compute_metrics(yt,pt,tte,thresh=t_tune))
        n_wf=sum(1 for b in banks if b['y_test'].sum()>0)
        rows.append(_row(en,'local_expert',bm,None,len(banks),n_wf,total_fraud))

    # ── MoE Gates ──
    gates=[StaticGate(),PerformanceGate(),ConfidenceGate(),TypologyAwareGate(),
           ConfidenceAwareGate(),ExpertChoiceGate(),MultiGateMoE()]
    for gate in gates:
        bm=[]; lf=[]
        for b in banks:
            bid=b['id']; yt=b['y_test']
            tte=b.get('typ_test',np.array(['IBM_Normal']*len(yt)))
            tvl=b.get('typ_val',np.array(['IBM_Normal']*len(b['y_val'])))
            vps=[]; tps=[]; vf1s=[]; avl=[]; tyf=[]

            for strat,(fl_model,bw,_) in fl_results.items():
                if strat=='persfl' and bid in bw:
                    pv=mlp_proba(bw[bid],b['X_val']); pt_p=mlp_proba(bw[bid],b['X_test'])
                else:
                    pv=mlp_proba(fl_model,b['X_val']); pt_p=mlp_proba(fl_model,b['X_test'])
                t_tune=tune_threshold(b['y_val'],pv)
                vps.append(pv); tps.append(pt_p)
                vf1s.append(fbeta_score(b['y_val'],(pv>=t_tune).astype(int),beta=2,zero_division=0)
                            if b['y_val'].sum()>0 else 0.)
                tyf.append(_typ_f1(b['y_val'],pv,tvl,t_tune) if b['y_val'].sum()>0 else {})
                avl.append(True)

            for en in EXPERT_NAMES:
                m=experts[bid].get(en)
                if m and m.trained:
                    pv=m.predict_proba(b['X_val']); pt_p=m.predict_proba(b['X_test'])
                    t_tune=tune_threshold(b['y_val'],pv)
                    vps.append(pv); tps.append(pt_p)
                    vf1s.append(fbeta_score(b['y_val'],(pv>=t_tune).astype(int),beta=2,zero_division=0)
                                if b['y_val'].sum()>0 else 0.)
                    tyf.append(_typ_f1(b['y_val'],pv,tvl,t_tune) if b['y_val'].sum()>0 else {})
                    avl.append(True)
                else:
                    vps.append(np.full(len(b['y_val']),0.5))
                    tps.append(np.full(len(yt),0.5))
                    vf1s.append(0.); tyf.append({}); avl.append(False)

            avl=np.array(avl); vf1s=np.array(vf1s)

            if isinstance(gate,TypologyAwareGate):
                gate.update(bid,tyf,avl,val_f1s=vf1s)
                ap=np.zeros(len(yt))
                for ut in np.unique(tte):
                    mask=tte==ut; w=gate.get_weights(n_exp,avl,bid=bid,txn_typ=ut)
                    for i,pt_p in enumerate(tps): ap[mask]+=w[i]*pt_p[mask]
            elif isinstance(gate,PerformanceGate):
                gate.update(bid,vf1s,avl); w=gate.get_weights(n_exp,avl,bid=bid)
                ap=sum(w[i]*tps[i] for i in range(n_exp))
            elif isinstance(gate,ConfidenceGate):
                gate.update(bid,vps,avl); w=gate.get_weights(n_exp,avl,bid=bid)
                ap=sum(w[i]*tps[i] for i in range(n_exp))
            else:
                w=gate.get_weights(n_exp,avl); ap=sum(w[i]*tps[i] for i in range(n_exp))

            if isinstance(gate,TypologyAwareGate):
                ap_val=np.zeros(len(b['y_val']))
                for ut in np.unique(tvl):
                    mask=tvl==ut; wv=gate.get_weights(n_exp,avl,bid=bid,txn_typ=ut)
                    for i,pv in enumerate(vps): ap_val[mask]+=wv[i]*pv[mask]
            else:
                if isinstance(gate,(PerformanceGate,ConfidenceGate)):
                    wv=gate.get_weights(n_exp,avl,bid=bid)
                else:
                    wv=gate.get_weights(n_exp,avl)
                ap_val=sum(wv[i]*vps[i] for i in range(n_exp))

            t_gate=tune_threshold(b['y_val'],ap_val)
            bm.append(compute_metrics(yt,ap,tte,thresh=t_gate)); lf.append(loc_f1.get(bid,0))

        n_wf=sum(1 for b in banks if b['y_test'].sum()>0)
        rows.append(_row(gate.name,'moe',bm,lf,len(banks),n_wf,total_fraud))
    return rows

# ================================================================
# COLORS — extended for new methods
# ================================================================
COLORS = {
    'fedavg':  '#4F46A0', 'fedprox': '#0F6E56', 'fednova': '#854F0B',
    'persfl':  '#185FA5', 'scaffold':'#C2410C', 'fedavgm': '#7C3AED',
    'xgb':     '#A32D2D', 'lgbm':    '#D85A30', 'catboost':'#F2A154',
    'rf':      '#059669', 'lr':      '#6B7280',
    'moe_static':'#AAAAAA','moe_performance':'#1D9E75',
    'moe_confidence':'#E04B8A','moe_typology_aware':'#9333EA',
    'moe_confidence_aware':'#0891B2','moe_expert_choice':'#65A30D','moe_multi_gate':'#DC2626',
}

def plot_results(df_bm, fl_hist, tag):
    strats=sorted(df_bm['strategy'].unique())
    fig,axes=plt.subplots(1,3,figsize=(22,6))
    fig.suptitle(f'{tag} — IBM AML · Temporal Split · Dirichlet\n'
                 'FL(6): FedAvg/FedProx/FedNova/PersFL/SCAFFOLD/FedAvgM  '
                 'ML(5): XGB/LGBM/CatBoost/RF/LR  MoE(4)',
                 fontsize=9, fontweight='bold')
    ax=axes[0]; x=np.arange(len(strats)); w=0.22
    for mi,(m,lbl,c) in enumerate([('auprc','AUPRC','#2E4057'),
                                    ('f2','F2','#048A81'),
                                    ('mcc','MCC','#54C6EB')]):
        vals=[float(df_bm[df_bm['strategy']==s][m].mean()) for s in strats]
        ax.bar(x+(mi-1)*w,vals,w,label=lbl,color=c,alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],fontsize=6,rotation=45,ha='right')
    ax.set_ylim(0,1); ax.legend(fontsize=8,frameon=False)
    ax.set_title('Primary Metrics (avg all banks)',fontsize=9,fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='y',alpha=0.25)
    ax=axes[1]
    for si,s in enumerate(strats):
        c=COLORS.get(s,'#999')
        ap=float(df_bm[df_bm['strategy']==s]['auprc'].mean())
        rc=float(df_bm[df_bm['strategy']==s]['recall'].mean())
        ax.bar(si-0.18,ap,0.35,color=c,alpha=0.85)
        ax.bar(si+0.18,rc,0.35,color=c,alpha=0.4,edgecolor=c,linewidth=1.5)
    ax.set_xticks(range(len(strats)))
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],fontsize=6,rotation=45,ha='right')
    ax.set_ylim(0,1); ax.set_title('AUPRC (solid) vs Recall (hollow)',fontsize=9,fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='y',alpha=0.25)
    ax=axes[2]
    dfh=pd.DataFrame(fl_hist)
    if not dfh.empty:
        for strat in FL_STRATEGIES:
            sub=dfh[dfh['strategy']==strat]
            if sub.empty: continue
            g=sub.groupby('round')['auprc'].mean().reset_index()
            ax.plot(g['round'],g['auprc'],label=strat.upper(),
                    color=COLORS.get(strat,'#999'),lw=2,marker='o',ms=4)
    ax.set_xlabel('FL Round'); ax.set_ylim(0,1)
    ax.set_title('FL Learning Curves (avg AUPRC)',fontsize=9,fontweight='bold')
    ax.legend(fontsize=7,frameon=False)
    ax.spines[['top','right']].set_visible(False); ax.grid(alpha=0.25)
    plt.tight_layout()
    p=f'{OUT}/{tag}_benchmark_results.png'
    plt.savefig(p,dpi=150,bbox_inches='tight'); plt.close(); print(f"Saved: {p}")

def print_summary(df_bm, label):
    print(f"\n{'='*70}")
    print(f"{label} — IBM AML · Temporal Split · Dirichlet")
    print("="*70)
    cols=['strategy','auprc','mcc','f2','f1','recall','specificity','fpr',
          'ap_at_100','typ_wf1','client_equity','worst_bank_f1','collab_gain',
          'threshold','n_eval_banks','n_banks_with_fraud','total_test_fraud']
    avail=[c for c in cols if c in df_bm.columns]
    print(df_bm[avail].sort_values('auprc',ascending=False).to_string(index=False))
    best=df_bm.loc[df_bm['auprc'].idxmax()]
    print(f"\nBEST: {best['strategy']}  AUPRC={best['auprc']:.4f}  MCC={best['mcc']:.4f}  F2={best['f2']:.4f}")


In [ ]:
# ================================================================
# MAIN LOOP — NB1: HI-Small + LI-Small × 3 alphas
# Total combos: 2 datasets × 3 alphas × (6 FL + 5 ML + 7 MoE) = 108
# ================================================================
all_combined = []

for DS in DATASETS:
    print(f"\n{'#'*65}")
    print(f"  DATASET: {DS}")
    print(f"{'#'*65}")

    df_raw = load_dataset(DS)
    X, y, typ, src, t_col = preprocess_ibm(df_raw)
    del df_raw; flush()
    print(f"  Preprocessed | {elapsed()}")

    ds_results = {}

    for alpha in ALPHAS:
        tag = f"{DS.replace('-','').lower()}_alpha{alpha}"
        print(f"\n{'='*55}")
        print(f"  {DS} | alpha={alpha} | {N_BANKS} banks")
        print(f"{'='*55}")

        banks = partition_dataset(X, y, typ, t_col, N_BANKS, alpha)

        # ── FL ──
        fl_results = {}; all_hist = []
        print("\nFL training:")
        for strat in FL_STRATEGIES:
            fl_model, bw, hist = run_fl(banks, strat, X.shape[1])
            if strat == 'persfl':
                init_X2 = np.vstack([b['X_train'][:50] for b in banks])
                init_y2 = np.concatenate([b['y_train'][:50] for b in banks])
                tm = init_mlp(init_X2, init_y2)
                bw_m = {bid: clone_mlp(tm, w) for bid, w in bw.items()}
                fl_results[strat] = (fl_model, bw_m, hist)
            else:
                fl_results[strat] = (fl_model, {}, hist)
            for h in hist: h['alpha'] = alpha; h['dataset'] = DS
            all_hist.extend(hist); flush()

        # ── ML ──
        print(f"\nLocal experts ({', '.join(EXPERT_NAMES)})...")
        experts = train_experts(banks); flush()

        # ── Evaluate ──
        print("\nEvaluating...")
        bm_rows = evaluate_all(banks, fl_results, experts)
        for r in bm_rows: r['dataset'] = DS; r['alpha'] = alpha

        df_bm = pd.DataFrame(bm_rows)
        df_bm.to_csv(f'{OUT}/{tag}_benchmark.csv', index=False)
        pd.DataFrame(all_hist).to_csv(f'{OUT}/{tag}_fl_history.csv', index=False)
        plot_results(df_bm, all_hist, tag)
        print_summary(df_bm, f"{DS} (alpha={alpha})")

        ds_results[tag] = df_bm
        all_combined.append(df_bm)
        del banks, fl_results, experts, all_hist; flush()
        print(f"  Done | {elapsed()}")

    # Per-dataset consolidated CSV
    ds_df = pd.concat(ds_results.values(), ignore_index=True)
    ds_df.to_csv(f'{OUT}/{DS.replace("-","").lower()}_all_alphas.csv', index=False)

    del X, y, typ, src, t_col; flush()
    print(f"\n{DS} complete | {elapsed()}")

# ── Final combined output ──
combined = pd.concat(all_combined, ignore_index=True)
combined.to_csv(f'{OUT}/nb1_all_datasets_combined.csv', index=False)

print(f"\n{'='*65}")
print("NB1 CONSOLIDATED — HI-Small + LI-Small × ALL ALPHAS")
print("="*65)
piv = combined.pivot_table(
    index=['dataset','alpha'], columns='strategy', values='auprc', aggfunc='mean'
).round(4)
print("\nAUPRC pivot (dataset × alpha × strategy):")
print(piv.to_string())

print(f"\nTotal runtime: {elapsed()}")
print("\nOutput files:")
for f in sorted(os.listdir(OUT)):
    if f.endswith(('.csv','.png')): print(f"  {f}")
